In [4]:
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")

b = spark.table("bronze_flights")
fmt = "yyyy-MM-dd HH:mmXXX"
s = (b
  .withColumn("scheduled_utc", F.to_timestamp("movement_scheduledTime_utc", fmt))
  .withColumn("scheduled_riyadh", F.expr("scheduled_utc + interval 3 hours"))
  .withColumn("movement_quality", F.concat_ws(",", "movement_quality"))
  .withColumnRenamed("codeshareStatus", "codeshare_status")
  .withColumnRenamed("isCargo", "is_cargo")
  .withColumnRenamed("callSign", "call_sign")
  .withColumnRenamed("aircraft_modeS", "aircraft_mode_s")
  .withColumnRenamed("movement_terminal", "terminal")
  .withColumnRenamed("movement_quality", "quality")
  .withColumnRenamed("movement_airport_timeZone", "other_airport_timezone")
  .withColumnRenamed("destination_airport_iata", "other_airport_iata")
  .withColumnRenamed("destination_airport_icao", "other_airport_icao")
  .withColumnRenamed("destination_airport_name", "other_airport_name")
  .drop("movement_scheduledTime_utc", "movement_scheduledTime_local",
        "origin_airport_name", "origin_airport_icao", "origin_airport_iata")
  .withColumn("airport_iata", F.lit("RUH"))
  .withColumn("is_canceled", F.col("status") == "Canceled"))

key = ["flight_number", "scheduled_utc", "flight_type"]
skip = {"quality", "_source_file", "_ingestion_time"}
score = sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)
            for c in s.columns if c not in skip)
w = Window.partitionBy(*key).orderBy(F.desc("_score"), F.desc("_ingestion_time"))
r = s.withColumn("_score", score).withColumn("_rn", F.row_number().over(w))

clean = r.filter("_rn = 1").drop("_score", "_rn").fillna({"terminal": "Unknown"})
dups  = r.filter("_rn > 1").drop("_rn")

clean.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("silver_flights_clean")
dups.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("quarantine_duplicates")

StatementMeta(, f92c2c89-6079-4835-991a-f926454bfdee, 6, Finished, Available, Finished, False)

In [5]:
print("silver:", spark.table("silver_flights").count())
print("quarantine:", spark.table("quarantine_duplicates").count())
print("تواريخ فاشلة:", spark.table("silver_flights").filter("scheduled_utc IS NULL").count())

StatementMeta(, f92c2c89-6079-4835-991a-f926454bfdee, 7, Finished, Available, Finished, False)

silver: 150519
quarantine: 2789
تواريخ فاشلة: 0
